# Phase 4C: reliability finalization

This notebook preserves the completed Phase 4/4B artifacts and finalizes the
reliability experiment without implementing MPC. Thresholds and persistence
settings are selected using **clean validation trajectories only**; the unseen
test trajectories are opened only for the final matrix.

The fixed design under test is deliberately small: a guarded signed-residual
sensor monitor, an H=10 rolling multi-step consistency monitor, and three
diagnostic states. MC Dropout is retained only as an ablation baseline.

In [1]:
import json
import sys
from dataclasses import replace
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from motor_model import DCMotorParams, simulate_motor
from reliability import (
    REDESIGN_STATE_NAMES,
    calibrate_cusum,
    cusum_monitor,
    guarded_predict,
    load_lstm_model,
    mc_dropout_predict,
    multistep_error,
    persistent_alarm,
    temporal_residual_features,
)

SEED = 2026
FAULT_START = 4.0
FAULT_END = 5.5
SENSOR_ENTER = 3
SENSOR_EXIT = 5
MODEL_ENTER = 3
MODEL_EXIT = 5
EWMA_ALPHA = 0.10
ROLLING_WINDOW = 20
FINAL_HORIZON = 10
TARGET_COMPONENT_VALIDATION_FAR = 0.001
MODEL_THRESHOLD_MARGIN = 1.05

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

data = np.load(project_root / "data" / "processed" / "dc_motor_lstm_dataset.npz")
model, model_config = load_lstm_model(
    project_root / "results" / "lstm_model_weights.pt",
    project_root / "results" / "configs" / "lstm_model_config.json",
)
normalization = model_config["normalization"]
input_mean = np.asarray(normalization["input_mean"])
input_std = np.asarray(normalization["input_std"])
target_mean = float(normalization["target_mean"][0])
target_std = float(normalization["target_std"][0])
window_length = int(model_config["window_length"])
timestep = float(data["timestep"])
time = data["time"].astype(float)
target_time = time[window_length:]
validation_ids = data["split_run_ids_validation"].astype(int)
test_ids = data["split_run_ids_test"].astype(int)

assert set(validation_ids).isdisjoint(test_ids)
print(f"Clean validation runs: {validation_ids.tolist()}")
print(f"Unseen test runs: {test_ids.tolist()}")

Clean validation runs: [3, 9, 11, 16, 19, 23]
Unseen test runs: [8, 15, 17, 22, 25, 28]


## 1. Preserve and review Phase 4 evidence

Phase 4B already established that MC Dropout reacts to operating-point/noisy-input
effects rather than the intended plant mismatch. Those files are read, not changed.
Final outputs in this notebook use a new `reliability_final_*` prefix.

In [2]:
metrics_dir = project_root / "results" / "metrics"
plots_dir = project_root / "results" / "plots"
configs_dir = project_root / "results" / "configs"
raw_dir = project_root / "results" / "raw"
old_config = json.loads((configs_dir / "reliability_config.json").read_text())
redesign_config = json.loads((configs_dir / "reliability_redesign_config.json").read_text())
prior_decision = json.loads((metrics_dir / "reliability_architecture_decision.json").read_text())
mc_evidence = pd.read_csv(metrics_dir / "reliability_mc_dropout_study.csv")
prior_ablation = pd.read_csv(metrics_dir / "reliability_redesign_ablation.csv")

model_mc = mc_evidence[mc_evidence["scenario"].isin(["load_step", "parameter_variation"])]
mc_model_auc = float(model_mc["raw_auc_vs_clean"].mean())
evidence_df = pd.DataFrame([
    {"evidence": "MC model-case AUC", "value": mc_model_auc},
    {"evidence": "H=15 load AUC", "value": 0.9092874807899248},
    {"evidence": "H=15 parameter AUC", "value": 0.6920785696685198},
    {"evidence": "Phase 4B clean test FAR", "value": prior_decision["clean_test_false_alarm_rate"]},
    {"evidence": "Phase 4B final F1", "value": float(prior_ablation.iloc[-1]["f1"])},
])
display(evidence_df.round(4))
print("MC Dropout retained in final core:", False)

,evidence,value
0,MC model-case AUC,0.3256
1,H=15 load AUC,0.9093
2,H=15 parameter AUC,0.6921
3,Phase 4B clean test FAR,0.0327
4,Phase 4B final F1,0.5293


MC Dropout retained in final core: False


## 2. Held-out scenarios and common helpers

Bias is an additive percentage of the dataset sensor full scale. Gradual drift is
a deliberately slow ramp from 0 to ±15% full scale over the final eight seconds.
The voltage-change case is a negative control: the changed voltage is supplied to
the LSTM, so it should remain NORMAL.

In [3]:
scenario_rng = np.random.default_rng(SEED + 900)
validation_voltage = data["voltage"][validation_ids].astype(float)
validation_measured = data["y_measured"][validation_ids].astype(float)
base_voltage = data["voltage"][test_ids].astype(float)
base_true = data["y_true"][test_ids].astype(float)
base_measured = data["y_measured"][test_ids].astype(float)
full_scale = float(np.max(np.abs(data["y_true"])))
fault_mask = time >= FAULT_START
output_fault_mask = target_time >= FAULT_START


def make_windows(voltage, measured):
    features = np.column_stack((voltage, measured))
    values = np.stack([
        features[start : start + window_length]
        for start in range(len(features) - window_length)
    ])
    return ((values - input_mean) / input_std).astype(np.float32)


@torch.no_grad()
def deterministic_rows(voltage, measured, batch_size=512):
    values = np.concatenate([make_windows(v, y) for v, y in zip(voltage, measured)])
    predicted = []
    for start in range(0, len(values), batch_size):
        predicted.append(model(torch.from_numpy(values[start : start + batch_size])).cpu().numpy()[:, 0])
    return (np.concatenate(predicted) * target_std + target_mean).reshape(len(voltage), -1)


def sampled_signal(values):
    return lambda instant: float(np.interp(instant, time, values))


def simulate_changed(voltage_rows, kind):
    nominal = DCMotorParams()
    shifted = replace(
        nominal,
        resistance=1.15 * nominal.resistance,
        inductance=0.85 * nominal.inductance,
        back_emf_constant=1.15 * nominal.back_emf_constant,
        torque_constant=0.85 * nominal.torque_constant,
        inertia=1.15 * nominal.inertia,
        viscous_friction=1.15 * nominal.viscous_friction,
        coulomb_friction=1.15 * nominal.coulomb_friction,
    )
    rows = []
    for voltage_values in voltage_rows:
        voltage = sampled_signal(voltage_values)
        if kind == "parameter":
            _, pre = simulate_motor(
                voltage, lambda _: 0.03, simulation_time=FAULT_START,
                timestep=timestep, params=nominal,
            )
            _, post = simulate_motor(
                lambda instant: voltage(instant + FAULT_START), lambda _: 0.03,
                simulation_time=float(data["duration"]) - FAULT_START,
                timestep=timestep, params=shifted, initial_state=tuple(pre[-1]),
            )
            states = np.vstack((pre, post[1:]))
        else:
            load = (
                (lambda instant: 0.03 if instant < FAULT_START else 0.15)
                if kind == "load" else (lambda _: 0.03)
            )
            _, states = simulate_motor(
                voltage, load, simulation_time=float(data["duration"]),
                timestep=timestep, params=nominal,
            )
        rows.append(states[:, 1])
    return np.asarray(rows)


def add_noise(values):
    return values + scenario_rng.normal(0.0, float(data["speed_noise_std"]), values.shape)


load_true = simulate_changed(base_voltage, "load")
parameter_true = simulate_changed(base_voltage, "parameter")
shifted_voltage = base_voltage.copy()
shifted_voltage[:, fault_mask] = np.clip(shifted_voltage[:, fault_mask] + 2.0, 0.0, 12.0)
voltage_true = simulate_changed(shifted_voltage, "voltage")

scenarios = {}


def add_scenario(name, title, voltage, true, measured, expected_state, active=fault_mask):
    scenarios[name] = {
        "title": title,
        "voltage": np.asarray(voltage),
        "true": np.asarray(true),
        "measured": np.asarray(measured),
        "expected_state": expected_state,
        "active": np.broadcast_to(active, np.asarray(measured).shape).copy(),
    }


add_scenario("clean", "Clean held-out operation", base_voltage, base_true, base_measured, 0, False)
gaussian = base_measured.copy()
gaussian[:, fault_mask] += scenario_rng.normal(0.0, 2.0, gaussian[:, fault_mask].shape)
add_scenario("gaussian_noise", "Added Gaussian sensor noise", base_voltage, base_true, gaussian, 1)
for percent in (5, 10, 15):
    measured = base_measured.copy()
    measured[:, fault_mask] += percent / 100 * full_scale
    add_scenario(f"bias_pos_{percent}", f"+{percent}% full-scale sensor bias", base_voltage, base_true, measured, 1)
negative_bias = base_measured.copy()
negative_bias[:, fault_mask] -= 0.05 * full_scale
add_scenario("bias_neg_5", "-5% full-scale sensor bias", base_voltage, base_true, negative_bias, 1)
ramp = np.clip((time - FAULT_START) / (time[-1] - FAULT_START), 0.0, 1.0)
for sign, name in ((1, "drift_positive"), (-1, "drift_negative")):
    measured = base_measured + sign * ramp[None, :] * 0.15 * full_scale
    add_scenario(name, name.replace("_", " ").title(), base_voltage, base_true, measured, 1)
dropout_active = (time >= FAULT_START) & (time < FAULT_END)
dropout = base_measured.copy()
dropout[:, dropout_active] = 0.0
add_scenario("sensor_dropout", "Temporary sensor dropout", base_voltage, base_true, dropout, 1, dropout_active)
add_scenario("load_step", "Unseen load-torque step", base_voltage, load_true, add_noise(load_true), 2)
add_scenario("parameter_variation", "Moderate parameter shift", base_voltage, parameter_true, add_noise(parameter_true), 2)
add_scenario("voltage_change", "Known +2 V operating-point change", shifted_voltage, voltage_true, add_noise(voltage_true), 0)
combined = add_noise(load_true)
combined[:, fault_mask] += 0.05 * full_scale
add_scenario("combined_bias_load", "+5% sensor bias plus load step", base_voltage, load_true, combined, 1)

final_scenario_names = [
    "clean", "gaussian_noise", "bias_pos_5", "bias_pos_10", "bias_pos_15",
    "drift_positive", "sensor_dropout", "load_step", "parameter_variation",
    "combined_bias_load",
]
print(f"Full scale: {full_scale:.3f} rad/s; +5% bias: {0.05 * full_scale:.3f} rad/s")
print("Final matrix:", final_scenario_names)

Full scale: 62.487 rad/s; +5% bias: 3.124 rad/s
Final matrix: ['clean', 'gaussian_noise', 'bias_pos_5', 'bias_pos_10', 'bias_pos_15', 'drift_positive', 'sensor_dropout', 'load_step', 'parameter_variation', 'combined_bias_load']


## 3. Sensor signal: instantaneous residual, signed EWMA, and two-sided CUSUM

The instantaneous guard is deliberately conservative (clean 99.9th percentile).
CUSUM uses the standard 0.5σ allowance and a clean-only percentile grid. A sample
can protect the autoregressive history immediately, but the diagnostic state needs
three consecutive abnormal samples to enter and five healthy samples to exit.

In [4]:
validation_prediction = deterministic_rows(validation_voltage, validation_measured)
validation_residual = validation_measured[:, window_length:] - validation_prediction
residual_center = float(np.median(validation_residual))
instant_threshold = float(np.percentile(np.abs(validation_residual), 99.9))

validation_temporal = [
    temporal_residual_features(row - residual_center, ROLLING_WINDOW, EWMA_ALPHA)
    for row in validation_residual
]
tau_signed_ewma = float(np.percentile(
    np.abs(np.concatenate([item["signed_ewma"] for item in validation_temporal])), 99.9
))
tau_absolute_ewma = float(np.percentile(
    np.concatenate([item["ewma"] for item in validation_temporal]), 99.9
))

sensor_calibration_rows = []
guarded_validation_by_percentile = {}
for percentile in (90.0, 95.0, 97.5, 99.0, 99.5, 99.9):
    candidate = calibrate_cusum(validation_residual, percentile, allowance_sigma=0.5)
    guarded_validation = guarded_predict(
        model, validation_voltage, validation_measured, normalization,
        window_length, instant_threshold, candidate,
        enter_count=SENSOR_ENTER, exit_count=SENSOR_EXIT,
    )
    guarded_validation_by_percentile[percentile] = guarded_validation
    sensor_calibration_rows.append({
        "percentile": percentile,
        "instant_gate": instant_threshold,
        "cusum_threshold": candidate["threshold"],
        "validation_false_alarm_rate": float(np.mean(guarded_validation["sensor_alarm"])),
    })
sensor_calibration_df = pd.DataFrame(sensor_calibration_rows)
eligible = sensor_calibration_df[
    sensor_calibration_df["validation_false_alarm_rate"] <= TARGET_COMPONENT_VALIDATION_FAR
]
selected_sensor_row = (eligible.iloc[0] if len(eligible) else sensor_calibration_df.iloc[-1])
sensor_percentile = float(selected_sensor_row["percentile"])
sensor_cusum = calibrate_cusum(validation_residual, sensor_percentile, allowance_sigma=0.5)
validation_guarded = guarded_validation_by_percentile[sensor_percentile]
display(sensor_calibration_df.round(5))
print("Selected sensor percentile:", sensor_percentile)

deterministic = {}
guarded = {}
feedback_history = {}
sensor_alarm = {}
sensor_features = {}
always_sensor_alarm = {}
for name, scenario in scenarios.items():
    prediction = deterministic_rows(scenario["voltage"], scenario["measured"])
    residual = scenario["measured"][:, window_length:] - prediction
    deterministic[name] = {"prediction": prediction, "residual": residual}
    monitors = [cusum_monitor(row, **sensor_cusum) for row in residual]
    always_sensor_alarm[name] = persistent_alarm(
        (np.abs(residual) > instant_threshold)
        | np.stack([monitor["score"] > sensor_cusum["threshold"] for monitor in monitors]),
        SENSOR_ENTER, SENSOR_EXIT,
    )
    guarded[name] = guarded_predict(
        model, scenario["voltage"], scenario["measured"], normalization,
        window_length, instant_threshold, sensor_cusum,
        enter_count=SENSOR_ENTER, exit_count=SENSOR_EXIT,
    )
    sensor_alarm[name] = guarded[name]["sensor_alarm"]
    feedback = scenario["measured"].copy()
    feedback[:, window_length:] = np.where(
        guarded[name]["substituted"], guarded[name]["prediction"],
        scenario["measured"][:, window_length:],
    )
    feedback_history[name] = feedback
    sensor_features[name] = [
        temporal_residual_features(row - residual_center, ROLLING_WINDOW, EWMA_ALPHA)
        for row in guarded[name]["residual"]
    ]

sensor_comparison_rows = []
for name in ["clean", "gaussian_noise", "bias_pos_5", "bias_pos_10", "bias_pos_15", "bias_neg_5", "drift_positive", "drift_negative", "sensor_dropout"]:
    scenario = scenarios[name]
    active = scenario["active"][:, window_length:]
    mask = np.ones_like(active) if name == "clean" else active
    residual = deterministic[name]["residual"]
    signed_ewma = np.stack([item["signed_ewma"] for item in sensor_features[name]])
    absolute_ewma = np.stack([item["ewma"] for item in sensor_features[name]])
    cusum_score = guarded[name]["cusum_score"]
    detectors = {
        "instantaneous": np.abs(residual) > instant_threshold,
        "signed_EWMA": np.abs(signed_ewma) > tau_signed_ewma,
        "absolute_EWMA": absolute_ewma > tau_absolute_ewma,
        "two_sided_CUSUM": cusum_score > sensor_cusum["threshold"],
        "final_guarded_persistent": sensor_alarm[name],
    }
    for detector, alarm in detectors.items():
        sensor_comparison_rows.append({
            "scenario": name,
            "detector": detector,
            "alarm_rate": float(np.mean(alarm[mask])),
        })
sensor_comparison_df = pd.DataFrame(sensor_comparison_rows)
display(sensor_comparison_df.pivot(index="scenario", columns="detector", values="alarm_rate").round(3))

feedback_rows = []
for name in ["clean", "bias_pos_5", "bias_pos_10", "bias_pos_15", "drift_positive", "sensor_dropout"]:
    active = scenarios[name]["active"][:, window_length:]
    mask = np.ones_like(active) if name == "clean" else active
    feedback_rows.append({
        "scenario": name,
        "always_measured_alarm_rate": float(np.mean(always_sensor_alarm[name][mask])),
        "virtual_feedback_alarm_rate": float(np.mean(sensor_alarm[name][mask])),
        "predicted_feedback_fraction": float(np.mean(guarded[name]["substituted"][mask])),
    })
feedback_df = pd.DataFrame(feedback_rows)
display(feedback_df.round(4))

,percentile,instant_gate,cusum_threshold,validation_false_alarm_rate
0,90.0,0.93721,0.52031,0.05151
1,95.0,0.93721,0.66666,0.02329
2,97.5,0.93721,0.82089,0.00988
3,99.0,0.93721,1.03490,0.00564
4,99.5,0.93721,1.24897,0.00423
5,99.9,0.93721,1.56968,0.00071


Selected sensor percentile: 99.9


detector,absolute_EWMA,final_guarded_persistent,instantaneous,signed_EWMA,two_sided_CUSUM
scenario,,,,,
bias_neg_5,0.802,0.788,0.006,0.801,0.794
bias_pos_10,0.632,0.607,0.012,0.622,0.612
bias_pos_15,0.902,0.891,0.016,0.899,0.893
bias_pos_5,0.448,0.426,0.006,0.441,0.432
clean,0.022,0.023,0.001,0.030,0.025
drift_negative,0.005,0.004,0.002,0.010,0.009
drift_positive,0.032,0.034,0.002,0.034,0.036
gaussian_noise,0.998,0.993,0.681,0.970,0.996
sensor_dropout,1.000,0.987,0.098,1.000,1.000


,scenario,always_measured_alarm_rate,virtual_feedback_alarm_rate,predicted_feedback_fraction
0,clean,0.0078,0.0226,0.0240
1,bias_pos_5,0.0464,0.4259,0.4295
2,bias_pos_10,0.1067,0.6069,0.6103
3,bias_pos_15,0.2141,0.8906,0.8937
4,drift_positive,0.0221,0.0343,0.0364
5,sensor_dropout,0.9867,0.9867,1.0000


## 4. Model mismatch: recursive multi-step consistency

H=10/RMS/rolling mean is selected before opening the test results: it is the
longest requested horizon, emphasizes sustained errors, and costs less than the
previous H=15 signal. The table still reports H=3, 5, and 10 with MAE, RMS,
normalized RMS, and rolling mean/RMS aggregation.

In [5]:
def multistep_rows(voltage, history, observed, horizon, metric):
    return np.stack([
        multistep_error(
            model, v, h, normalization, window_length, horizon,
            observed=o, metric=metric,
        )
        for v, h, o in zip(voltage, history, observed)
    ])


def rolling_aggregate(scores, mode="mean", count=ROLLING_WINDOW):
    output = np.full_like(scores, np.nan)
    for row_index, row in enumerate(scores):
        first = np.flatnonzero(np.isfinite(row))[0]
        values = row[first:]
        values = values if mode == "mean" else values**2
        cumulative = np.r_[0.0, np.cumsum(values)]
        divisor = np.minimum(np.arange(1, len(values) + 1), count)
        starts = np.maximum(0, np.arange(len(values)) - count + 1)
        aggregated = (cumulative[1:] - cumulative[starts]) / divisor
        output[row_index, first:] = aggregated if mode == "mean" else np.sqrt(aggregated)
    return output


study_names = ["clean", "load_step", "parameter_variation", "voltage_change", "bias_pos_5", "drift_positive"]
multistep_cache = {}
horizon_rows = []
for horizon in (3, 5, 10):
    for metric in ("mae", "rms"):
        validation_raw = multistep_rows(
            validation_voltage, validation_measured, validation_measured, horizon, metric
        )
        multistep_cache[("validation", horizon, metric)] = validation_raw
        for name in study_names:
            scenario = scenarios[name]
            multistep_cache[(name, horizon, metric)] = multistep_rows(
                scenario["voltage"], feedback_history[name], scenario["measured"], horizon, metric
            )
        for aggregation in ("mean", "rms"):
            validation_score = rolling_aggregate(validation_raw, aggregation)
            clean_values = validation_score[np.isfinite(validation_score)]
            normalization_scale = float(np.median(clean_values))
            for reported_metric in ([metric] if metric == "mae" else ["rms", "normalized_rms"]):
                scale = normalization_scale if reported_metric == "normalized_rms" else 1.0
                for name in study_names:
                    score = rolling_aggregate(multistep_cache[(name, horizon, metric)], aggregation) / scale
                    active = scenarios[name]["active"][:, window_length:]
                    mask = np.isfinite(score) & (active if name != "clean" else True)
                    values = score[mask]
                    reference = clean_values / scale
                    auc = roc_auc_score(
                        np.r_[np.zeros(len(reference)), np.ones(len(values))],
                        np.r_[reference, values],
                    )
                    horizon_rows.append({
                        "horizon": horizon,
                        "metric": reported_metric,
                        "rolling": aggregation,
                        "scenario": name,
                        "auc_vs_clean_validation": float(auc),
                        "mean_score": float(np.mean(values)),
                    })
horizon_study_df = pd.DataFrame(horizon_rows)
display(
    horizon_study_df[
        (horizon_study_df["metric"] == "rms")
        & (horizon_study_df["rolling"] == "mean")
        & horizon_study_df["scenario"].isin(["load_step", "parameter_variation", "voltage_change", "bias_pos_5"])
    ].pivot(index="horizon", columns="scenario", values="auc_vs_clean_validation").round(3)
)

scenario,bias_pos_5,load_step,parameter_variation,voltage_change
horizon,,,,
3,0.768,0.655,0.559,0.513
5,0.772,0.711,0.586,0.516
10,0.773,0.845,0.643,0.511


## 5. Final model score and clean-only calibration

The final score is a positive CUSUM of normalized rolling H=10 RMS error.
This converts the useful distribution shift into a persistent change signal.
Whenever the sensor is suspect, the model CUSUM is reset and held at zero;
a 0.30 s cooldown covers the forecast and rolling windows after recovery.

In [6]:
validation_h10_raw = multistep_cache[("validation", FINAL_HORIZON, "rms")]
validation_h10_rolling = rolling_aggregate(validation_h10_raw, "mean")
model_scale = float(np.median(validation_h10_rolling[np.isfinite(validation_h10_rolling)]))
validation_normalized = validation_h10_rolling / model_scale
model_center = float(np.median(validation_normalized[np.isfinite(validation_normalized)]))
model_allowance = 0.1 * float(np.std(validation_normalized[np.isfinite(validation_normalized)], ddof=1))


def extend_alarm(alarm, count):
    extended = alarm.copy()
    for offset in range(1, count + 1):
        extended[:, offset:] |= alarm[:, :-offset]
    return extended


def cumulative_mismatch(values, blocked=None):
    blocked = np.zeros_like(values, dtype=bool) if blocked is None else blocked
    result = np.full_like(values, np.nan)
    for row_index, row in enumerate(values):
        accumulator = 0.0
        for index, value in enumerate(row):
            if not np.isfinite(value):
                continue
            if blocked[row_index, index]:
                accumulator = 0.0
                result[row_index, index] = 0.0
            else:
                accumulator = max(0.0, accumulator + value - model_center - model_allowance)
                result[row_index, index] = accumulator
    return result


validation_blocked = extend_alarm(
    validation_guarded["sensor_alarm"], FINAL_HORIZON + ROLLING_WINDOW
)
validation_mismatch = cumulative_mismatch(validation_normalized, validation_blocked)
model_calibration_rows = []
for percentile in (90.0, 95.0, 97.5, 99.0, 99.5, 99.9, 100.0):
    threshold = float(np.percentile(validation_mismatch[np.isfinite(validation_mismatch)], percentile))
    alarm = persistent_alarm(
        np.nan_to_num(validation_mismatch > threshold), MODEL_ENTER, MODEL_EXIT
    )
    model_calibration_rows.append({
        "percentile": percentile,
        "threshold": threshold,
        "validation_false_alarm_rate": float(np.mean(alarm)),
    })
model_calibration_df = pd.DataFrame(model_calibration_rows)
eligible = model_calibration_df[
    model_calibration_df["validation_false_alarm_rate"] <= TARGET_COMPONENT_VALIDATION_FAR
]
selected_model_row = (eligible.iloc[0] if len(eligible) else model_calibration_df.iloc[-1])
model_percentile = float(selected_model_row["percentile"])
model_threshold = MODEL_THRESHOLD_MARGIN * float(selected_model_row["threshold"])
display(model_calibration_df.round(5))
print("Selected model percentile and held-out safety margin:", model_percentile, MODEL_THRESHOLD_MARGIN)

mismatch_raw = {}
mismatch_rolling = {}
mismatch_score = {}
model_alarm = {}
for name, scenario in scenarios.items():
    cache_key = (name, FINAL_HORIZON, "rms")
    if cache_key not in multistep_cache:
        multistep_cache[cache_key] = multistep_rows(
            scenario["voltage"], feedback_history[name], scenario["measured"],
            FINAL_HORIZON, "rms",
        )
    mismatch_raw[name] = multistep_cache[cache_key]
    mismatch_rolling[name] = rolling_aggregate(mismatch_raw[name], "mean") / model_scale
    blocked = extend_alarm(sensor_alarm[name], FINAL_HORIZON + ROLLING_WINDOW)
    mismatch_score[name] = cumulative_mismatch(mismatch_rolling[name], blocked)
    model_alarm[name] = persistent_alarm(
        np.nan_to_num(mismatch_score[name] > model_threshold), MODEL_ENTER, MODEL_EXIT
    )

model_rows = []
clean_model_values = mismatch_score["clean"][np.isfinite(mismatch_score["clean"])]
for name in ["clean", "load_step", "parameter_variation", "voltage_change", "bias_pos_5", "drift_positive"]:
    active = scenarios[name]["active"][:, window_length:]
    mask = np.isfinite(mismatch_score[name]) & (active if name != "clean" else True)
    values = mismatch_score[name][mask]
    auc = 0.5 if name == "clean" else roc_auc_score(
        np.r_[np.zeros(len(clean_model_values)), np.ones(len(values))],
        np.r_[clean_model_values, values],
    )
    model_rows.append({
        "scenario": name,
        "alarm_rate": float(np.mean(model_alarm[name][active])) if active.any() else float(np.mean(model_alarm[name])),
        "AUC_vs_clean": float(auc),
    })
model_study_df = pd.DataFrame(model_rows)
display(model_study_df.round(4))

,percentile,threshold,validation_false_alarm_rate
0,90.0,41.50961,0.09893
1,95.0,62.75870,0.04996
2,97.5,78.10455,0.02540
3,99.0,94.15982,0.01030
4,99.5,99.77264,0.00536
5,99.9,104.30363,0.00141
6,100.0,104.60933,0.00000


Selected model percentile and held-out safety margin: 100.0 1.05


,scenario,alarm_rate,AUC_vs_clean
0,clean,0.0028,0.5000
1,load_step,0.5268,0.9074
2,parameter_variation,0.0343,0.6190
3,voltage_change,0.0000,0.4491
4,bias_pos_5,0.0000,0.3408
5,drift_positive,0.0000,0.5380


## 6. Final three-state diagnostic matrix

Sensor precedence is retained because a suspect sensor directly contaminates the
mismatch observation. The combined-fault row is intentionally strict: with only
one speed sensor, a bias that begins with a plant shift can remain observationally
ambiguous. No unsupported fourth state is added.

In [7]:
def detection_latency(flagged, active):
    latencies = []
    for row_flagged, row_active in zip(flagged, active):
        detections = np.flatnonzero(row_flagged & row_active)
        if len(detections):
            latencies.append(float(target_time[detections[0]] - FAULT_START))
    return float(np.mean(latencies)) if latencies else np.nan


def binary_summary(labels, predictions):
    labels = np.asarray(labels, dtype=bool).ravel()
    predictions = np.asarray(predictions, dtype=bool).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "false_alarm_rate": float(np.mean(predictions[~labels])) if (~labels).any() else np.nan,
    }


codes = {}
expected = {}
scenario_rows = []
for name, scenario in scenarios.items():
    codes[name] = np.where(
        sensor_alarm[name], 1, np.where(model_alarm[name], 2, 0)
    ).astype(np.int8)
    active = scenario["active"][:, window_length:]
    expected[name] = np.where(active, scenario["expected_state"], 0).astype(np.int8)
    flagged = codes[name] != 0
    labels = expected[name] != 0
    summary = binary_summary(labels, flagged)
    clean_mask = (
        np.ones_like(active) if name == "clean"
        else np.broadcast_to(target_time < FAULT_START, active.shape)
    )
    recovery_mask = (
        np.broadcast_to(target_time >= FAULT_END, active.shape)
        if name == "sensor_dropout" else None
    )
    scenario_rows.append({
        "scenario": name,
        "detection_rate": float(np.mean(flagged[active])) if active.any() else np.nan,
        "precision": summary["precision"],
        "recall": summary["recall"],
        "f1": summary["f1"],
        "false_alarm_rate": float(np.mean(flagged[clean_mask])),
        "detection_latency_seconds": detection_latency(flagged, active) if active.any() else np.nan,
        "diagnostic_state_accuracy": float(np.mean(codes[name][active] == expected[name][active])) if active.any() else float(np.mean(codes[name] == 0)),
        "post_recovery_alarm_rate": float(np.mean(flagged[recovery_mask])) if recovery_mask is not None else np.nan,
    })
scenario_metrics_df = pd.DataFrame(scenario_rows)
display(scenario_metrics_df[scenario_metrics_df["scenario"].isin(final_scenario_names)].round(4))

final_expected = np.concatenate([expected[name].ravel() for name in final_scenario_names])
final_codes = np.concatenate([codes[name].ravel() for name in final_scenario_names])
global_binary = binary_summary(final_expected != 0, final_codes != 0)
global_state_accuracy = float(np.mean(final_expected == final_codes))
state_confusion = confusion_matrix(final_expected, final_codes, labels=np.arange(3))
print("Global binary:", global_binary)
print("Global diagnostic-state accuracy:", round(global_state_accuracy, 4))

,scenario,detection_rate,precision,recall,f1,false_alarm_rate,detection_latency_seconds,diagnostic_state_accuracy,post_recovery_alarm_rate
0,clean,NaN,0.0000,0.0000,0.0000,0.0254,NaN,0.9746,NaN
1,gaussian_noise,0.9929,0.9979,0.9929,0.9954,0.0044,0.045,0.9929,NaN
2,bias_pos_5,0.4259,0.9951,0.4259,0.5965,0.0044,0.020,0.4259,NaN
3,bias_pos_10,0.6069,0.9966,0.6069,0.7544,0.0044,0.020,0.6069,NaN
4,bias_pos_15,0.8906,0.9977,0.8906,0.9411,0.0044,0.020,0.8906,NaN
6,drift_positive,0.0343,0.9429,0.0343,0.0663,0.0044,4.350,0.0343,NaN
8,sensor_dropout,0.9867,0.3572,0.9867,0.5245,0.0044,0.020,0.9867,0.4066
9,load_step,0.5300,1.0000,0.5300,0.6928,0.0000,1.890,0.5268,NaN
10,parameter_variation,0.0366,0.9724,0.0366,0.0706,0.0022,2.860,0.0335,NaN
12,combined_bias_load,0.2468,1.0000,0.2468,0.3959,0.0000,0.020,0.0316,NaN


Global binary: {'precision': 0.9119215799336889, 'recall': 0.48231168039036293, 'f1': 0.6309280406921658, 'false_alarm_rate': 0.05816831683168317}
Global diagnostic-state accuracy: 0.6716


## 7. Ablation A–F

All six methods below are evaluated on the same final unseen-test matrix. Method B
runs MC Dropout only for this baseline; no final decision uses its output.

In [8]:
def mc_rows(voltage, measured, passes=10):
    values = np.concatenate([make_windows(v, y) for v, y in zip(voltage, measured)])
    mean, variance = mc_dropout_predict(model, values, passes=passes, batch_size=256)
    shape = (len(voltage), len(time) - window_length)
    return (
        (mean.numpy()[:, 0] * target_std + target_mean).reshape(shape),
        (variance.numpy()[:, 0] * target_std**2).reshape(shape),
    )


mc_baseline = {}
redesign_codes = {}
for name in final_scenario_names:
    scenario = scenarios[name]
    mc_mean, mc_variance = mc_rows(scenario["voltage"], scenario["measured"])
    mc_baseline[name] = {
        "residual": np.abs(scenario["measured"][:, window_length:] - mc_mean),
        "variance": mc_variance,
    }
    old_guarded = guarded_predict(
        model, scenario["voltage"], scenario["measured"], normalization,
        window_length, redesign_config["guard_threshold"], None,
    )
    old_monitors = [
        cusum_monitor(
            row, redesign_config["cusum_center"], redesign_config["cusum_allowance"],
            redesign_config["tau_cusum"],
        )
        for row in old_guarded["residual"]
    ]
    old_sensor = (
        np.stack([
            temporal_residual_features(row, ROLLING_WINDOW, redesign_config["ewma_alpha"])["ewma"]
            for row in old_guarded["residual"]
        ]) > redesign_config["tau_ewma"]
    ) | np.stack([monitor["score"] > redesign_config["tau_cusum"] for monitor in old_monitors])
    old_h15 = multistep_rows(
        scenario["voltage"], scenario["measured"], scenario["measured"],
        redesign_config["mismatch_horizon"], "rms",
    )
    old_mismatch = rolling_aggregate(old_h15, "mean") > redesign_config["tau_mismatch"]
    redesign_codes[name] = np.where(old_sensor, 1, np.where(old_mismatch, 2, 0)).astype(np.int8)

methods = {
    "A_fixed_residual": [],
    "B_residual_plus_MC": [],
    "C_Phase4B_redesign": [],
    "D_sensor_only": [],
    "E_multistep_only": [],
    "F_final_combined": [],
}
labels = []
clean_predictions = {name: [] for name in methods}
for name in final_scenario_names:
    labels.append((expected[name] != 0).ravel())
    predictions = {
        "A_fixed_residual": np.abs(deterministic[name]["residual"]) > old_config["tau_r"],
        "B_residual_plus_MC": (
            (mc_baseline[name]["residual"] > old_config["tau_r"])
            | (mc_baseline[name]["variance"] > old_config["tau_sigma"])
        ),
        "C_Phase4B_redesign": redesign_codes[name] != 0,
        "D_sensor_only": sensor_alarm[name],
        "E_multistep_only": model_alarm[name],
        "F_final_combined": codes[name] != 0,
    }
    for method, prediction in predictions.items():
        methods[method].append(prediction.ravel())
        if name == "clean":
            clean_predictions[method].append(prediction.ravel())
labels = np.concatenate(labels)
ablation_rows = []
for method, parts in methods.items():
    prediction = np.concatenate(parts)
    summary = binary_summary(labels, prediction)
    ablation_rows.append({
        "method": method,
        "detection_rate": float(np.mean(prediction[labels])),
        "precision": summary["precision"],
        "recall": summary["recall"],
        "f1": summary["f1"],
        "false_alarm_rate": summary["false_alarm_rate"],
        "clean_only_false_alarm_rate": float(np.mean(np.concatenate(clean_predictions[method]))),
    })
ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df.round(4))

,method,detection_rate,precision,recall,f1,false_alarm_rate,clean_only_false_alarm_rate
0,A_fixed_residual,0.1099,0.8955,0.1099,0.1958,0.0160,0.0134
1,B_residual_plus_MC,0.1550,0.8326,0.1550,0.2613,0.0389,0.0294
2,C_Phase4B_redesign,0.5983,0.8327,0.5983,0.6963,0.1501,0.0327
3,D_sensor_only,0.3876,0.8938,0.3876,0.5407,0.0575,0.0226
4,E_multistep_only,0.0948,0.9947,0.0948,0.1731,0.0006,0.0028
5,F_final_combined,0.4823,0.9119,0.4823,0.6309,0.0582,0.0254


## 8. Representative temporal traces

Scores are divided by their thresholds so 1.0 is always the alarm boundary.
The state panel makes debounce and precedence transitions explicit.

In [9]:
plots_dir.mkdir(parents=True, exist_ok=True)
representative = ["bias_pos_5", "drift_positive", "load_step", "combined_bias_load"]
for name in representative:
    scenario = scenarios[name]
    row = 0
    features = sensor_features[name][row]
    fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)
    axes[0].plot(target_time, scenario["true"][row, window_length:], label="true")
    axes[0].plot(target_time, scenario["measured"][row, window_length:], alpha=0.55, label="measured")
    axes[0].plot(target_time, guarded[name]["prediction"][row], "--", label="guarded LSTM")
    axes[0].set_ylabel("Speed (rad/s)")
    axes[0].legend(ncol=3)
    axes[1].plot(target_time, guarded[name]["residual"][row], label="signed residual")
    axes[1].axhline(instant_threshold, color="tab:red", linestyle="--")
    axes[1].axhline(-instant_threshold, color="tab:red", linestyle="--", label="instant gate")
    axes[1].set_ylabel("Residual")
    axes[1].legend()
    axes[2].plot(target_time, np.abs(features["signed_ewma"]) / tau_signed_ewma, label="|signed EWMA| / threshold")
    axes[2].plot(target_time, guarded[name]["cusum_score"][row] / sensor_cusum["threshold"], label="CUSUM / threshold")
    axes[2].axhline(1.0, color="tab:red", linestyle="--")
    axes[2].set_ylabel("Sensor score")
    axes[2].legend()
    axes[3].plot(target_time, mismatch_score[name][row] / model_threshold, label="multi-step M / threshold")
    axes[3].axhline(1.0, color="tab:red", linestyle="--")
    axes[3].set_ylabel("Model score")
    axes[3].legend()
    axes[4].step(target_time, codes[name][row], where="post")
    axes[4].set_yticks(range(3), REDESIGN_STATE_NAMES)
    axes[4].set_ylim(-0.25, 2.25)
    axes[4].set_ylabel("State")
    axes[4].set_xlabel("Time (s)")
    for axis in axes:
        axis.axvline(FAULT_START, color="black", linestyle=":")
        axis.grid(alpha=0.25)
    fig.suptitle(scenario["title"])
    fig.tight_layout()
    fig.savefig(plots_dir / f"reliability_final_{name}.png", dpi=160)
    plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_17484\3227298922.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_17484\3227298922.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_17484\3227298922.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_17484\3227298922.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. CPU cost per timestep

The timing includes one guarded one-step prediction, one H=10 recursive check,
and constant-memory temporal arithmetic. MC N=10 and saved H=15 cost are shown
only for comparison.

In [10]:
sample_window = torch.from_numpy(make_windows(base_voltage[0], base_measured[0])[:1])
future_voltage = torch.from_numpy(
    ((base_voltage[0, window_length : window_length + FINAL_HORIZON] - input_mean[0]) / input_std[0]).astype(np.float32)
)


def benchmark(function, repetitions):
    for _ in range(5):
        function()
    started = perf_counter()
    for _ in range(repetitions):
        function()
    return 1000 * (perf_counter() - started) / repetitions


@torch.no_grad()
def basic_step():
    model(sample_window)


def mc_step():
    mc_dropout_predict(model, sample_window, passes=10, batch_size=1)


@torch.no_grad()
def multistep_step():
    window = sample_window.clone()
    for step in range(FINAL_HORIZON):
        normalized_prediction = model(window)[:, 0]
        physical_prediction = normalized_prediction * target_std + target_mean
        normalized_speed = (physical_prediction - input_mean[1]) / input_std[1]
        next_sample = torch.stack((future_voltage[step : step + 1], normalized_speed), dim=1)
        window = torch.cat((window[:, 1:], next_sample[:, None, :]), dim=1)


def temporal_trace():
    temporal_residual_features(validation_residual[0], ROLLING_WINDOW, EWMA_ALPHA)
    cusum_monitor(validation_residual[0], **sensor_cusum)


basic_ms = benchmark(basic_step, 300)
mc_ms = benchmark(mc_step, 50)
multistep_ms = benchmark(multistep_step, 50)
temporal_ms = benchmark(temporal_trace, 100) / validation_residual.shape[1]
cost_df = pd.DataFrame([
    {"component": "basic_guarded_prediction", "milliseconds_per_step": basic_ms},
    {"component": "temporal_sensor_and_state", "milliseconds_per_step": temporal_ms},
    {"component": "multistep_h10", "milliseconds_per_step": multistep_ms},
    {"component": "final_combined", "milliseconds_per_step": basic_ms + temporal_ms + multistep_ms},
    {"component": "MC_dropout_n10_ablation", "milliseconds_per_step": mc_ms},
])
display(cost_df.round(4))

,component,milliseconds_per_step
0,basic_guarded_prediction,0.3455
1,temporal_sensor_and_state,0.0016
2,multistep_h10,3.9850
3,final_combined,4.3321
4,MC_dropout_n10_ablation,1.0324


## 10. Save final artifacts and make the engineering decision

Readiness is deliberately not averaged into a flattering single number. Every
stated gate must pass. Failure of slow drift or plant diagnosis therefore keeps
MPC integration at NO-GO even if gross sensor faults are excellent.

In [11]:
metric_lookup = scenario_metrics_df.set_index("scenario")
clean_far = float(metric_lookup.loc["clean", "false_alarm_rate"])
small_bias_rate = float(metric_lookup.loc["bias_pos_5", "detection_rate"])
drift_rate = float(metric_lookup.loc[["drift_positive", "drift_negative"], "detection_rate"].mean())
load_rate = float(metric_lookup.loc["load_step", "detection_rate"])
parameter_rate = float(metric_lookup.loc["parameter_variation", "detection_rate"])
gross_sensor_rate = float(metric_lookup.loc[["bias_pos_10", "bias_pos_15", "sensor_dropout"], "detection_rate"].mean())
load_auc = float(model_study_df.set_index("scenario").loc["load_step", "AUC_vs_clean"])
parameter_auc = float(model_study_df.set_index("scenario").loc["parameter_variation", "AUC_vs_clean"])
dropout_recovery_rate = float(metric_lookup.loc["sensor_dropout", "post_recovery_alarm_rate"])

small_bias_improved = small_bias_rate >= 1.5 * 0.2126508531002913
drift_improved = drift_rate >= 0.15
load_reliable = load_rate >= 0.80
parameter_meaningful = parameter_rate >= 0.50
ready_for_mpc = bool(
    clean_far <= 0.03
    and gross_sensor_rate >= 0.90
    and small_bias_improved
    and drift_improved
    and load_reliable
    and parameter_meaningful
    and load_auc > mc_model_auc
)

final_config = {
    "seed": SEED,
    "window_length": window_length,
    "sensor": {
        "instant_percentile": 99.9,
        "instant_threshold": instant_threshold,
        "cusum_percentile": sensor_percentile,
        **sensor_cusum,
        "signed_ewma_alpha": EWMA_ALPHA,
        "signed_ewma_threshold": tau_signed_ewma,
        "enter_count": SENSOR_ENTER,
        "exit_count": SENSOR_EXIT,
        "feedback": "predicted on instantaneous guard; debounced SENSOR_SUSPECT state",
    },
    "model": {
        "horizon": FINAL_HORIZON,
        "rolling_window": ROLLING_WINDOW,
        "metric": "normalized rolling RMS positive CUSUM",
        "scale": model_scale,
        "center": model_center,
        "allowance": model_allowance,
        "percentile": model_percentile,
        "threshold": model_threshold,
        "held_out_safety_margin": MODEL_THRESHOLD_MARGIN,
        "enter_count": MODEL_ENTER,
        "exit_count": MODEL_EXIT,
        "sensor_cooldown_samples": FINAL_HORIZON + ROLLING_WINDOW,
    },
    "state_codes": {name: int(code) for code, name in enumerate(REDESIGN_STATE_NAMES)},
    "sensor_precedence": True,
    "mc_dropout_retained": False,
    "validation_sensor_false_alarm_rate": float(selected_sensor_row["validation_false_alarm_rate"]),
    "validation_model_false_alarm_rate": float(selected_model_row["validation_false_alarm_rate"]),
}
decision = {
    "1_small_bias_improved": bool(small_bias_improved),
    "2_drift_improved": bool(drift_improved),
    "3_load_mismatch_reliable": bool(load_reliable),
    "4_parameter_variation_meaningful": bool(parameter_meaningful),
    "5_final_sensor_signal": "Conservative instantaneous guard plus signed-residual two-sided CUSUM, virtual predicted feedback, and 3/5 persistence.",
    "6_final_model_signal": "Sensor-masked positive CUSUM of normalized rolling H=10 recursive RMS prediction error.",
    "7_mc_dropout_retained": False,
    "8_final_states": REDESIGN_STATE_NAMES.tolist(),
    "9_clean_false_alarm_rate": clean_far,
    "10_ready_for_mpc": ready_for_mpc,
    "small_bias_detection_rate": small_bias_rate,
    "mean_drift_detection_rate": drift_rate,
    "load_detection_rate": load_rate,
    "parameter_detection_rate": parameter_rate,
    "load_auc": load_auc,
    "parameter_auc": parameter_auc,
    "dropout_post_recovery_alarm_rate": dropout_recovery_rate,
    "recommendation": "NO-GO: narrow the novelty to LSTM-MPC plus validated gross-sensor-fault fallback; keep model mismatch as monitoring-only unless an independent plant/sensor reference is added and revalidated. The virtual-feedback latch also needs an independent recovery reference before closed-loop use.",
}

metrics_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
raw_dir.mkdir(parents=True, exist_ok=True)
sensor_calibration_df.to_csv(metrics_dir / "reliability_final_sensor_calibration.csv", index=False)
model_calibration_df.to_csv(metrics_dir / "reliability_final_model_calibration.csv", index=False)
sensor_comparison_df.to_csv(metrics_dir / "reliability_final_sensor_study.csv", index=False)
horizon_study_df.to_csv(metrics_dir / "reliability_final_multistep_study.csv", index=False)
scenario_metrics_df.to_csv(metrics_dir / "reliability_final_scenario_metrics.csv", index=False)
ablation_df.to_csv(metrics_dir / "reliability_final_ablation.csv", index=False)
cost_df.to_csv(metrics_dir / "reliability_final_cost.csv", index=False)
(configs_dir / "reliability_final_config.json").write_text(json.dumps(final_config, indent=2), encoding="utf-8")
(metrics_dir / "reliability_final_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")

diagnostic_arrays = {
    "scenario_names": np.array(list(scenarios)),
    "state_names": REDESIGN_STATE_NAMES,
    "time": target_time.astype(np.float32),
}
for field, values in (
    ("measured", {name: scenario["measured"][:, window_length:] for name, scenario in scenarios.items()}),
    ("guarded_prediction", {name: guarded[name]["prediction"] for name in scenarios}),
    ("signed_residual", {name: guarded[name]["residual"] for name in scenarios}),
    ("cusum_sensor_score", {name: guarded[name]["cusum_score"] for name in scenarios}),
    ("mismatch_score", mismatch_score),
    ("diagnostic_codes", codes),
    ("expected_codes", expected),
):
    diagnostic_arrays[field] = np.stack([values[name] for name in scenarios])
np.savez_compressed(raw_dir / "reliability_final_diagnostics.npz", **diagnostic_arrays)

display(Markdown(f'''        1. **Did small-bias detection improve?** **{decision['1_small_bias_improved']}** - +5% detection is {small_bias_rate:.1%}, versus 21.3% in Phase 4B.
2. **Did drift detection improve?** **{decision['2_drift_improved']}** - mean positive/negative drift detection is only {drift_rate:.1%}; the clean-FAR-constrained signed innovation cannot reliably expose the slow ramp.
3. **Can load mismatch be detected reliably?** **{decision['3_load_mismatch_reliable']}** - AUC is {load_auc:.3f}, but state detection is only {load_rate:.1%}.
4. **Can parameter variation be detected meaningfully?** **{decision['4_parameter_variation_meaningful']}** - AUC {parameter_auc:.3f}, detection {parameter_rate:.1%}.
5. **Final sensor signal:** {decision['5_final_sensor_signal']}
6. **Final model signal:** {decision['6_final_model_signal']}
7. **MC Dropout retained?** **{decision['7_mc_dropout_retained']}**; ablation only.
8. **Final diagnostic states:** {', '.join(decision['8_final_states'])}.
9. **Clean false-alarm rate:** {clean_far:.2%}.
10. **Ready for MPC?** **{decision['10_ready_for_mpc']}**. Dropout post-recovery alarms remain {dropout_recovery_rate:.1%}. {decision['recommendation']}
'''))

        1. **Did small-bias detection improve?** **True** - +5% detection is 42.6%, versus 21.3% in Phase 4B.
2. **Did drift detection improve?** **False** - mean positive/negative drift detection is only 3.2%; the clean-FAR-constrained signed innovation cannot reliably expose the slow ramp.
3. **Can load mismatch be detected reliably?** **False** - AUC is 0.907, but state detection is only 53.0%.
4. **Can parameter variation be detected meaningfully?** **False** - AUC 0.619, detection 3.7%.
5. **Final sensor signal:** Conservative instantaneous guard plus signed-residual two-sided CUSUM, virtual predicted feedback, and 3/5 persistence.
6. **Final model signal:** Sensor-masked positive CUSUM of normalized rolling H=10 recursive RMS prediction error.
7. **MC Dropout retained?** **False**; ablation only.
8. **Final diagnostic states:** NORMAL, SENSOR_SUSPECT, MODEL_MISMATCH.
9. **Clean false-alarm rate:** 2.54%.
10. **Ready for MPC?** **False**. Dropout post-recovery alarms remain 40.7%. NO-GO: narrow the novelty to LSTM-MPC plus validated gross-sensor-fault fallback; keep model mismatch as monitoring-only unless an independent plant/sensor reference is added and revalidated. The virtual-feedback latch also needs an independent recovery reference before closed-loop use.


In [12]:
required_outputs = [
    configs_dir / "reliability_final_config.json",
    metrics_dir / "reliability_final_decision.json",
    metrics_dir / "reliability_final_scenario_metrics.csv",
    metrics_dir / "reliability_final_ablation.csv",
    raw_dir / "reliability_final_diagnostics.npz",
]
assert all(path.exists() for path in required_outputs)
assert all((configs_dir / name).exists() for name in [
    "reliability_config.json", "reliability_redesign_config.json",
])
assert (metrics_dir / "reliability_architecture_decision.json").exists()
assert final_config["mc_dropout_retained"] is False
assert set(np.unique(np.concatenate([value.ravel() for value in codes.values()]))) <= {0, 1, 2}
assert float(selected_sensor_row["validation_false_alarm_rate"]) <= 0.002
assert float(selected_model_row["validation_false_alarm_rate"]) <= 0.002
assert not model.training
print("Phase 4C checks passed; prior Phase 4/4B files remain intact and MPC was not implemented.")

Phase 4C checks passed; prior Phase 4/4B files remain intact and MPC was not implemented.
